In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# ====================================================
# 1. KHỞI TẠO SPARK KẾT NỐI VỚI CLUSTER
# ====================================================
print("🚀 Đang khởi động Spark kết nối Cluster...")
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Cluster_Process_Articles_Multimodal") \
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "3g") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Đã kết nối Spark thành công!")

🚀 Đang khởi động Spark kết nối Cluster...


26/03/25 15:01:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Đã kết nối Spark thành công!


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType, DateType

# ====================================================
# 2. KHAI BÁO ĐƯỜNG DẪN HDFS
# ====================================================
INPUT_PATH = "hdfs://namenode:9000/data/raw/"
OUTPUT_PATH = "hdfs://namenode:9000/data/processed/"

print("⏳ Đang đọc transactions_train.csv từ HDFS...")

try:
    # 1. Đọc dữ liệu (Không dùng inferSchema để tiết kiệm RAM, ta sẽ ép kiểu thủ công)
    transactions = spark.read.csv(INPUT_PATH + "transactions_train.csv", header=True)

    # 2. ÉP KIỂU & LÀM SẠCH (Cực kỳ quan trọng để giảm bộ nhớ)
    # t_dat: String -> Date
    # article_id: Phải dùng lpad để khớp với file articles (10 chữ số)
    # price: String -> Float
    print("⚙️ Đang chuẩn hóa định dạng dữ liệu...")
    
    df_processed = transactions.withColumn("t_dat", F.to_date(F.col("t_dat"))) \
                               .withColumn("article_id", F.lpad(F.col("article_id"), 10, "0")) \
                               .withColumn("price", F.col("price").cast(FloatType())) \
                               .withColumn("sales_channel_id", F.col("sales_channel_id").cast("int"))

    # 3. FEATURE ENGINEERING (Ví dụ: Tính tổng chi tiêu của mỗi khách hàng)
    # Bước này Spark sẽ chạy cực nhanh nhờ cơ chế Shuffle 200 partitions bạn đã config
    print("📊 Đang tính toán các chỉ số bổ sung...")
    
    # Tính tổng tiền mỗi khách hàng đã chi
    customer_spending = df_processed.groupBy("customer_id") \
                                    .agg(F.sum("price").alias("total_spent"), 
                                         F.count("article_id").alias("total_items"))

    # 4. LỌC DỮ LIỆU (Ví dụ: Chỉ lấy giao dịch từ năm 2020 để giảm tải)
    df_recent = df_processed.filter(F.col("t_dat") >= "2020-01-01")

    # 5. XEM THỬ KẾT QUẢ
    print("✨ Dữ liệu sau khi xử lý (Top 5):")
    df_recent.show(5)

    # ====================================================
    # 3. GHI KẾT QUẢ XUỐNG HDFS (Dạng Parquet)
    # ====================================================
    # Ghi Parquet sẽ giúp file 3GB CSV giảm xuống còn khoảng vài trăm MB
    print(f"📤 Đang ghi kết quả vào: {OUTPUT_PATH}transactions_cleaned.parquet")
    
    df_recent.write.mode("overwrite").parquet(OUTPUT_PATH + "transactions_cleaned.parquet")
    
    print("✅ Hoàn tất xử lý file giao dịch!")

except Exception as e:
    print(f"❌ Lỗi: {str(e)}")

⏳ Đang đọc transactions_train.csv từ HDFS...


⚙️ Đang chuẩn hóa định dạng dữ liệu...
📊 Đang tính toán các chỉ số bổ sung...
✨ Dữ liệu sau khi xử lý (Top 5):


KeyboardInterrupt: 